In CPlantBox, plant growth is genetically predetermined for optimal conditions according to the structural input parameters described in notebook 1. However, different environmental conditions can lead to adaptations in the plant phenotype. Such adaptations in root development are taken into account in CPlantBox by root growth inhibited due to carbon deficiency, changes in root distribution in the soil profile due to root tropism, or changes in root growth angles and branching patterns.
In this section we show how to represent different root trait responses to soil and plant conditions. For demonstration purposes, the examples assume static soil and boundary conditions.

Tropism is an other example of plant plasticity. It was presented in notebook 1.


### 0. Loading libraries

In [ ]:
# generic Python libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, display # to show svg files in the notebook

# specific CPlantBox libraries
search_path = '../lib'
sys.path.append(search_path)
import plantbox as pb #CPlantBox itself
import visualisation.vtk_plot as vp # for quick vizualisations

### 1. Scaling root elongation

In the following example, we demonstrate how to use CPlantBox to model the impact of soil conditions (i.e., varying soil strength) on root elongation rates.\
ATT: the resulting scaling should be between [0 and 1].\
ATT: the resulting scaling should cover the whole area that the roots reach.

In [ ]:
plant = pb.MappedPlant()
path = "../modelparameter/structural/rootsystem/"
filename = "Anagallis_femina_Leitner_2010"
plant.readParameters(path + filename + ".xml")

scaling = pb.EquidistantGrid1D(0, -50, 100) # (min_depth, max_depth, resolution)
soil_strength = np.ones((100,))
soil_strength[60:80] = 0.2 # empirical soil strength value

assert (len(soil_strength) == len(scaling.data)) 
scaling.data = soil_strength  # set proportionality factors
print("-5 cm ", scaling.getValue(pb.Vector3d(0, 0, -5)))
# print("-15 cm", scaling.getValue(pb.Vector3d(0, 0, -__ )))  # TODO: print the value of the scale elongation at 15cm of depth

for p in plant.getOrganRandomParameter(pb.root):
    p.f_se = scaling  # apply the scaling to all organs

plant.initialize()
sim_time = 30.0  # days
dt = 0.1 

for i in range(0, round(sim_time / dt)):  # Simulation
    # option for dynamic scale update
    # scaling.data = soil_strength  
    plant.simulate(dt, False)
    
plant.write("results/example_scale_elongation.vtp")
_  = vp.plot_roots(plant, "creationTime", interactiveImage = False) # Plot, using vtk. This function can take some time for large plants.

**Q: appart from soil strength (or density), what else soil values could be used to scale the growth rates?**
**Q: In the code above, uncomment the line with 'TODO' and replace ___ to print the value of the soil strength at depth 15cm.**\
**Q: Change the function describing the soil resistance to root growth: instead of an exponential function, use (for example) a cubing function.**\
**Q: Currently, we get a different soil_strength (5 instead of 1) between depth 30cm and 40cm. Change the code to have a soil strength of 1 everywhere, and a soil strength of 0.1 between depth 5 and 10.**\
**Q: change the resolution of the scaling, from 0.5cm to 1cm.**\
**Q: ADVANCED: change the soil strength value according to time. E.g., set the initial soil strength to 1 everywhere and make it decrease with time until it reaches 0.**

The code below also scales a plant function according to a soil value.\
**Q: set part of the soil_strength in the code below and above to 0. Observe the effect this has on the resulting root system. Can you guess what the f_sbp parameter scales?**

In [ ]:
plant = pb.MappedPlant()
path = "../modelparameter/structural/rootsystem/"
filename = "Anagallis_femina_Leitner_2010"
plant.readParameters(path + filename + ".xml")

scaling = pb.EquidistantGrid1D(0, -50, 100) # (min_depth, max_depth, resolution)
soil_strength = np.ones((100,))
soil_strength[10:30] = 0. # empirical soil strength value

assert (len(soil_strength) == len(scaling.data)) 
scaling.data = soil_strength  # set proportionality factors
print("-5 cm ", scaling.getValue(pb.Vector3d(0, 0, -5)))
# print("-15 cm", scaling.getValue(pb.Vector3d(0, 0, -__ )))  # TODO: print the value of the scale elongation at 15cm of depth

for p in plant.getOrganRandomParameter(pb.root):
    p.f_sbp = scaling  # apply the scaling to all organs

plant.initialize()
sim_time = 30.0  # days
dt = 0.1 

for i in range(0, round(sim_time / dt)):  # Simulation
    # option for dynamic scale update
    # scaling.data = soil_strength  
    plant.simulate(dt, False)
    
plant.write("results/example_scale_branching.vtp")
_  = vp.plot_roots(plant, "creationTime", interactiveImage = False) # Plot, using vtk. This function can take some time for large plants.